In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agriculture-climate-slm-challenge/test_questions.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/sample_submission.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/train_qa.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/documents.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/dataset-metadata.json
/kaggle/input/competitions/agriculture-climate-slm-challenge/baseline_submission.csv


In [2]:
import os
import pandas as pd

# To find the file paths dynamically from /kaggle/input
train_path = None
test_path = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        if filename == 'train_qa.csv':
            train_path = full_path
        elif filename == 'test_questions.csv':
            test_path = full_path

print(f"Train path: {train_path}")
print(f"Test path: {test_path}")

# Loading CSV files
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Displaying top of files
print("\n--- Train Head ---")
display(train_df.head())

print("\n--- Test Head ---")
display(test_df.head())

Train path: /kaggle/input/competitions/agriculture-climate-slm-challenge/train_qa.csv
Test path: /kaggle/input/competitions/agriculture-climate-slm-challenge/test_questions.csv

--- Train Head ---


,question,topic,crop,agro_zone,document_id,reference_answer,QuestionId
0,Weevils in stored maize without chemicals?,post_harvest,maize,semi_arid,doc_pos_001,Dry to twelve to thirteen percent and seal in ...,1
1,Insurance paid but my field still failed — why?,climate_adaptation,general,semi_arid,doc_cli_003,Basis risk means index payouts may not match i...,2
2,Which cover crop helps between maize seasons?,soil_health,general,sub_humid,doc_soi_002,Mucuna or lablab reduce erosion and suppress w...,3
3,Maize stalks lodging before harvest — nutrient...,fertiliser,maize,sub_humid,doc_fer_001,Ensure balanced NPK including potassium for st...,4
4,What are signs of bean rust?,crop_diseases,beans,highland,doc_dis_002,Reddish-brown pustules on leaf undersides in c...,5



--- Test Head ---


,QuestionId,question,topic,crop,agro_zone
0,1001,How should I apply nitrogen to leaching-prone ...,crop_diseases,maize,sub_humid
1,1002,Grass gone in August — feed strategy?,livestock,livestock,semi_arid
2,1003,Red spots under my bean leaves during the rains.,crop_diseases,beans,highland
3,1004,Fresh cow dung on vegetable beds — safe?,fertiliser,general,sub_humid
4,1005,How much compost per hectare?,fertiliser,general,sub_humid


In [3]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Train TF-IDF with word AND char-level n-grams
# Char n-grams capture spelling variations, suffixes, and punctuation patterns
word_vec = TfidfVectorizer(ngram_range=(1, 3), stop_words='english', sublinear_tf=True)
char_vec = TfidfVectorizer(ngram_range=(3, 5), analyzer='char_wb', sublinear_tf=True)

# Combine textual fields
train_text = train_df['question'].fillna('').astype(str)
test_text = test_df['question'].fillna('').astype(str)

tfidf_w_train = word_vec.fit_transform(train_text)
tfidf_w_test = word_vec.transform(test_text)

tfidf_c_train = char_vec.fit_transform(train_text)
tfidf_c_test = char_vec.transform(test_text)

# 2. Similarity matrix (weighted word + char similarity)
sim_word = cosine_similarity(tfidf_w_test, tfidf_w_train)
sim_char = cosine_similarity(tfidf_c_test, tfidf_c_train)
base_sim = 0.6 * sim_char + 0.4 * sim_word

# 3. Apply Metadata Multipliers
best_indices = []
for i in range(len(test_df)):
    row_test = test_df.iloc[i]
    scores = base_sim[i].copy()
    
    # Metadata matching bonuses
    for j in range(len(train_df)):
        row_train = train_df.iloc[j]
        bonus = 1.0
        
        # Crop match
        if str(row_test.get('crop', '')).lower() == str(row_train.get('crop', '')).lower() and row_test.get('crop'):
            bonus += 0.35
        # Agro-zone match
        if str(row_test.get('agro_zone', '')).lower() == str(row_train.get('agro_zone', '')).lower() and row_test.get('agro_zone'):
            bonus += 0.25
        # Topic match
        if str(row_test.get('topic', '')).lower() == str(row_train.get('topic', '')).lower() and row_test.get('topic'):
            bonus += 0.20
            
        scores[j] *= bonus
        
    best_indices.append(scores.argmax())

test_df['Answer'] = train_df['reference_answer'].iloc[best_indices].values

In [4]:
# Format submission
submission = test_df[['QuestionId', 'Answer']].copy()
submission['Answer'] = submission['Answer'].fillna("No answer provided.")

# Save directly to /kaggle/working/submission.csv
submission.to_csv("/kaggle/working/submission.csv", index=False)

# Verification checks
print(f"Saved: /kaggle/working/submission.csv")
print(f"Row count: {len(submission)} (expected: 12)")
print(f"Columns: {list(submission.columns)} (expected: ['QuestionId', 'Answer'])")
print(f"Null answers: {submission['Answer'].isna().sum()}")
print(f"Empty answers: {(submission['Answer'].str.strip() == '').sum()}")

display(submission)

Saved: /kaggle/working/submission.csv
Row count: 12 (expected: 12)
Columns: ['QuestionId', 'Answer'] (expected: ['QuestionId', 'Answer'])
Null answers: 0
Empty answers: 0


,QuestionId,Answer
0,1001,Only if rain or irrigation is imminent; otherw...
1,1002,Use short-season drought-tolerant varieties al...
2,1003,"Use clean seed, avoid dusk overhead irrigation..."
3,1004,Dry to about twelve to thirteen percent moistu...
4,1005,"It should be dark, crumbly, and free of undeco..."
5,1006,Dry quickly on raised mats and remove mouldy k...
6,1007,Acidity binds phosphorus and limits nodulation...
7,1008,Stem borer tunneling kills the growing point i...
8,1009,"It should be dark, crumbly, and free of undeco..."
9,1010,"Harvest forage at boot stage, sun-dry on racks..."
